# 08 — Données temporelles

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- manipuler `date`, `time` et `datetime` ;
- calculer une durée avec `timedelta` ;
- attacher un fuseau horaire avec **`zoneinfo`** (la lib standard Python, on ne met plus `pytz`) ;
- formater / parser une date avec `strftime` et `strptime` ;
- comprendre UTC et l'importance des datetimes *aware*.

## Prérequis

- types primitifs, conditions, boucles ;
- f-strings ;
- listes, tuples, dictionnaires, ensembles.

Pas encore vus :

- `def`, type hints ;
- exceptions personnalisées.

## Plan

1. Les trois types principaux
2. Date courante, maintenant
3. Construire une date / un datetime
4. Attributs (`year`, `hour`, …)
5. `timedelta` : durées et arithmétique
6. Formater avec `strftime`
7. Parser avec `strptime`
8. Fuseaux horaires avec `zoneinfo`
9. Naïf vs aware
10. Synthèse
11. Exercices

---


## 1. Les trois types principaux

| Type | Usage |
|---|---|
| `date` | une date civile : année / mois / jour |
| `time` | une heure : heure / minute / seconde |
| `datetime` | date + heure combinées (le plus utilisé) |

Ils vivent dans le module `datetime` (attention au piège : le module et la classe portent le même nom).

In [11]:
from datetime import date, time, datetime, timedelta

---


## 2. Date courante, maintenant

In [ ]:
date.today()

In [ ]:
datetime.now()

⚠️ `datetime.now()` renvoie un datetime **naïf** : sans fuseau horaire. Nous verrons plus bas comment attacher un fuseau proprement.

### `datetime.utcnow()` est déprécié en Python 3.12+

Préférer `datetime.now(tz=...)` avec un fuseau explicite.

---


## 3. Construire une date / un datetime

In [ ]:
anniversaire = date(1990, 5, 17)
print(anniversaire)

In [ ]:
rdv = datetime(2026, 4, 14, 9, 30, 0)
print(rdv)

In [ ]:
midi = time(12, 0, 0)
print(midi)

---


## 4. Attributs

In [12]:
rdv = datetime(2026, 4, 14, 9, 30, 0)

In [ ]:
rdv.year

In [ ]:
rdv.month

In [ ]:
rdv.day

In [ ]:
rdv.hour

In [ ]:
rdv.minute

In [ ]:
rdv.weekday()  # 0 = lundi, 6 = dimanche

In [ ]:
rdv.isoweekday()  # 1 = lundi, 7 = dimanche

---


## 5. `timedelta` : durées et arithmétique

`timedelta` représente une **durée**. C'est ce qu'on obtient en soustrayant deux datetimes.

In [3]:
duree = timedelta(days=7)
print(duree)

NameError: name 'timedelta' is not defined

In [ ]:
timedelta(hours=2, minutes=30)

In [ ]:
date(2026, 1, 1) + timedelta(days=365)

In [1]:
datetime(2026, 4, 14) - datetime(2026, 1, 1)

NameError: name 'datetime' is not defined

Le résultat d'une soustraction est un `timedelta`. On peut récupérer les jours avec `.days`.

In [ ]:
ecart = datetime(2026, 4, 14) - datetime(2026, 1, 1)
print('jours :', ecart.days)

---


## 6. Formater avec `strftime`

`strftime` (*string format time*) convertit un `datetime` en chaîne selon un format.

In [21]:
rdv = datetime(2026, 4, 14, 9, 30, 0)

In [ ]:
rdv.strftime('%Y-%m-%d')

In [ ]:
rdv.strftime('%d/%m/%Y %H:%M')

In [ ]:
rdv.strftime('%A %d %B %Y')  # nom du jour et du mois (locale par défaut)

### Codes de format courants

| Code | Signification | Exemple |
|---|---|---|
| `%Y` | Année sur 4 chiffres | `2026` |
| `%y` | Année sur 2 chiffres | `26` |
| `%m` | Mois (01-12) | `04` |
| `%d` | Jour (01-31) | `14` |
| `%H` | Heure 24h (00-23) | `09` |
| `%M` | Minutes | `30` |
| `%S` | Secondes | `00` |
| `%A` | Nom du jour | `mardi` |
| `%B` | Nom du mois | `avril` |
| `%w` | Numéro du jour (0 = dimanche) | `2` |
| `%j` | Jour de l'année | `104` |

---


## 7. Parser avec `strptime`

`strptime` fait l'inverse : il lit une chaîne et construit un `datetime`. On lui donne la chaîne **et** le format attendu.

In [22]:
datetime.strptime('2026-04-14', '%Y-%m-%d')

datetime.datetime(2026, 4, 14, 0, 0)

In [ ]:
datetime.strptime('14/04/2026 09:30', '%d/%m/%Y %H:%M')

Si la chaîne ne correspond pas au format, `strptime` lève `ValueError`.

In [20]:
try:
    datetime.strptime('pas une date', '%Y-%m-%d')
except ValueError as err:
    print('erreur :', err)

erreur : time data 'pas une date' does not match format '%Y-%m-%d'


### `fromisoformat` / `isoformat`

Pour le format ISO 8601 (`2026-04-14T09:30:00`), plus rapide à écrire.

In [ ]:
datetime.fromisoformat('2026-04-14T09:30:00')

In [ ]:
datetime(2026, 4, 14, 9, 30).isoformat()

---


## 8. Fuseaux horaires avec `zoneinfo`

Depuis Python 3.9, la bibliothèque standard fournit **`zoneinfo`**. Plus besoin de `pytz` : on utilise les fuseaux IANA (`'Europe/Paris'`, `'America/New_York'`, etc.).

In [10]:
from zoneinfo import ZoneInfo

In [ ]:
paris = ZoneInfo('Europe/Paris')
now_paris = datetime.now(paris)
print(now_paris)

In [ ]:
ny = ZoneInfo('America/New_York')
datetime.now(ny)

### Conversion entre fuseaux

In [31]:
rdv_paris = datetime(2026, 4, 14, 9, 30, tzinfo=ZoneInfo('Europe/Paris'))
rdv_paris

datetime.datetime(2026, 4, 14, 9, 30, tzinfo=zoneinfo.ZoneInfo(key='Europe/Paris'))

In [ ]:
rdv_paris.astimezone(ZoneInfo('America/New_York'))

In [ ]:
rdv_paris.astimezone(ZoneInfo('UTC'))

### Lister les fuseaux disponibles

In [7]:
from zoneinfo import available_timezones
noms = sorted(available_timezones())
print(len(noms), 'fuseaux disponibles')
print(noms[:5])

498 fuseaux disponibles
['Africa/Abidjan', 'Africa/Accra', 'Africa/Addis_Ababa', 'Africa/Algiers', 'Africa/Asmara']


---


## 9. Naïf vs aware

- **Naïf** : datetime sans `tzinfo`. On ne sait pas s'il est en heure locale, UTC, autre…
- **Aware** : datetime avec un `tzinfo` attaché. Il représente un instant non ambigu.

**Règle d'or** : dès qu'un datetime a vocation à voyager (base de données, JSON, API), il doit être aware, idéalement en UTC.

In [14]:
naif = datetime(2026, 4, 14, 9, 30)
print(naif.tzinfo)

None


In [13]:
aware = datetime(2026, 4, 14, 9, 30, tzinfo=ZoneInfo('Europe/Paris'))
print(aware.tzinfo)

Europe/Paris


⚠️ On ne peut pas soustraire un naïf et un aware : Python lève `TypeError`.

In [ ]:
try:
    naif - aware
except TypeError as err:
    print('erreur :', err)

---


## 10. Synthèse

| Besoin | Outil |
|---|---|
| Date civile | `date(y, m, d)` |
| Horodatage complet | `datetime(y, m, d, h, mi, s)` |
| Durée | `timedelta(days=..., hours=...)` |
| Maintenant local | `datetime.now(ZoneInfo('Europe/Paris'))` |
| Maintenant UTC | `datetime.now(ZoneInfo('UTC'))` |
| Format affichage | `dt.strftime('%d/%m/%Y')` |
| Parser chaîne | `datetime.strptime(s, format)` |
| ISO 8601 | `.isoformat()` / `.fromisoformat()` |
| Changer de fuseau | `dt.astimezone(ZoneInfo('...'))` |

### Règles à retenir

1. Préférer **`zoneinfo`** à `pytz` (stdlib, plus simple).
2. Toute date destinée à voyager doit être **aware**.
3. Stocker en **UTC**, afficher en fuseau local.
4. `strptime`/`strftime` sont symétriques : format = même chaîne.

---


## 11. Exercices

### Exercice 1 — Ton âge en jours *(facile)*

Demander à l'utilisateur sa date de naissance au format `jj/mm/aaaa`, puis afficher son âge en jours.

In [25]:
date_naissance_str = input("Entrez votre date de naissance au format jj/mm/aaaa : ")

date_naissance = datetime.strptime(date_naissance_str, "%d/%m/%Y").date()
aujourd_hui = date.today()

age_en_jours = (aujourd_hui - date_naissance).days
print(age_en_jours)

8388


In [26]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=1)


📝 Exercice 1 marqué comme tenté 🟢


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import date, datetime

texte = input('Date de naissance (jj/mm/aaaa) : ')
naissance = datetime.strptime(texte, '%d/%m/%Y').date()
age_jours = (date.today() - naissance).days
print('âge en jours :', age_jours)
```

</details>

### Exercice 2 — Jour de la semaine *(facile)*

Demander une date au format `jj/mm/aaaa` et afficher le jour de la semaine correspondant (en français, avec `%A` si votre locale le permet, sinon via une liste).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import datetime

jours = ['lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi', 'samedi', 'dimanche']
texte = input('Date (jj/mm/aaaa) : ')
d = datetime.strptime(texte, '%d/%m/%Y')
print(jours[d.weekday()])
```

</details>

### Exercice 3 — Dans N jours *(facile)*

Demander un nombre `n` à l'utilisateur et afficher la date dans `n` jours.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import date, timedelta

n = int(input('Nombre de jours : '))
cible = date.today() + timedelta(days=n)
print(cible.isoformat())
```

</details>

### Exercice 4 — Décompte avant un événement *(moyen)*

Demander à l'utilisateur une date d'événement au format ISO (`aaaa-mm-jj`), afficher le nombre de jours restants. Si l'événement est passé, afficher un message spécifique.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import date

texte = input('Date (aaaa-mm-jj) : ')
cible = date.fromisoformat(texte)
ecart = (cible - date.today()).days
if ecart > 0:
    print('J-', ecart)
elif ecart == 0:
    print("c'est aujourd'hui !")
else:
    print('événement passé il y a', -ecart, 'jours')
```

</details>

### Exercice 5 — Heure à New York *(moyen)*

Afficher l'heure courante à Paris et à New York, dans le format `HH:MM` et avec les noms des fuseaux.

In [29]:
# Votre code ici

paris = datetime.now(ZoneInfo('Europe/Paris'))
new_york = datetime.now(ZoneInfo('America/New_York'))

print('Paris     :', paris.strftime('%H:%M'))
print('New York :', new_york.strftime('%H:%M'))


Paris     : 09:05
New York : 03:05


In [30]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=5)


📝 Exercice 5 marqué comme tenté 🟢


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import datetime
from zoneinfo import ZoneInfo

paris = datetime.now(ZoneInfo('Europe/Paris'))
ny = datetime.now(ZoneInfo('America/New_York'))

print('Europe/Paris     :', paris.strftime('%H:%M'))
print('America/New_York :', ny.strftime('%H:%M'))
```

</details>

### Exercice 6 — Durée d'un film *(moyen)*

Demander une heure de début (`HH:MM`) et une durée en minutes, afficher l'heure de fin (`HH:MM`, sur 24h).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import datetime, timedelta

debut_txt = input('Heure de début (HH:MM) : ')
duree = int(input('Durée (minutes) : '))

debut = datetime.strptime(debut_txt, '%H:%M')
fin = debut + timedelta(minutes=duree)
print('fin :', fin.strftime('%H:%M'))
```

</details>

### Exercice 7 — Tous les lundis de l'année *(difficile)*

Afficher toutes les dates de lundis de l'année 2026, une par ligne, au format `jj/mm/aaaa`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=7)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import date, timedelta

d = date(2026, 1, 1)
# décaler jusqu'au premier lundi
while d.weekday() != 0:
    d = d + timedelta(days=1)

while d.year == 2026:
    print(d.strftime('%d/%m/%Y'))
    d = d + timedelta(days=7)
```

</details>

### Exercice 8 — Réunion trans-fuseau *(difficile)*

Un formateur à Paris organise une réunion à 9h30 (heure de Paris) le 14 avril 2026. Afficher l'heure correspondante pour trois stagiaires à :
- New York (`America/New_York`)
- Tokyo (`Asia/Tokyo`)
- Sydney (`Australia/Sydney`)
Au format `fuseau : jj/mm/aaaa HH:MM`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="08_Donnees_temporelles", exercice=8)


<details>
<summary>📖 Voir la correction</summary>

```python
from datetime import datetime
from zoneinfo import ZoneInfo

rdv = datetime(2026, 4, 14, 9, 30, tzinfo=ZoneInfo('Europe/Paris'))

for fuseau in ('America/New_York', 'Asia/Tokyo', 'Australia/Sydney'):
    local = rdv.astimezone(ZoneInfo(fuseau))
    print(fuseau, ':', local.strftime('%d/%m/%Y %H:%M'))
```

</details>

---


## Ressources externes

- [`datetime` — doc Python](https://docs.python.org/3/library/datetime.html)
- [`zoneinfo` — doc Python](https://docs.python.org/3/library/zoneinfo.html)
- [PEP 615 — `zoneinfo` dans la stdlib](https://peps.python.org/pep-0615/)
- [Liste IANA des fuseaux](https://www.iana.org/time-zones)